In [ ]:
!pip -q install datasets

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

In [ ]:
dataset = load_dataset("JSSICE/Multi-Domain-Sentiment-Dataset") ## 1                                                 ###

README.md:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


domain_sentiment_data.tar/sorted_data_ac(…): reconstructing file:   0%|          |  0.00B /  101MB            

domain_sentiment_data.tar/sorted_data_ac(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11 [00:00<?, ? examples/s]

In [ ]:
print(dataset)
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['__key__', '__url__', 'review'],
        num_rows: 11
    })
})
['__key__', '__url__', 'review']


In [ ]:
for i in range(len(dataset["train"])):
    print(i, dataset["train"][i]["__key__"])

0 sorted_data_acl/dvd/negative
1 sorted_data_acl/dvd/unlabeled
2 sorted_data_acl/dvd/positive
3 sorted_data_acl/kitchen_&_housewares/negative
4 sorted_data_acl/kitchen_&_housewares/unlabeled
5 sorted_data_acl/kitchen_&_housewares/positive
6 sorted_data_acl/books/negative
7 sorted_data_acl/books/positive
8 sorted_data_acl/electronics/negative
9 sorted_data_acl/electronics/unlabeled
10 sorted_data_acl/electronics/positive


# Extract Labeled Reviews into a Clean **DataFrame**

In [ ]:
import re
import pandas as pd

# I got help from ChatGPT to pull the reviews out of the dataset.
# I ran the code myself and checked the extracted data.
rows = []

for item in dataset["train"]:
    key = item["__key__"]

    if not (key.endswith("/positive") or key.endswith("/negative")):
        continue

    parts = key.split("/")
    domain = parts[-2]
    sentiment = parts[-1]

    raw_text = item["review"].decode("utf-8", errors="ignore")

    reviews = re.findall(
        r"<review_text>(.*?)</review_text>",
        raw_text,
        flags=re.DOTALL
    )

    for review in reviews:
        rows.append({
            "review_text": review.strip(),
            "sentiment": sentiment,
            "domain": domain
        })
                                                   ######2
df = pd.DataFrame(rows)

print(df.head())
print("\nTotal reviews:", len(df))

                                         review_text sentiment domain
0  This entire movie could have run in only 20 mi...  negative    dvd
1  If you are looking for a good movie to buy for...  negative    dvd
2  Fun to watch but my 9 1/2 yr. old daughter cou...  negative    dvd
3  Every review I've read about this movie pops a...  negative    dvd
4  The acting was very good. The pace was adequat...  negative    dvd

Total reviews: 8000


In [ ]:
print(df.isnull().sum())                           ###2
print("Duplicate reviews:", df.duplicated(subset=["review_text"]).sum())

review_text    0
sentiment      0
domain         0
dtype: int64
Duplicate reviews: 147


In [ ]:
duplicates = df[df.duplicated(subset=["review_text"], keep=False)]
conflicts = duplicates.groupby("review_text").agg({
    "sentiment": "nunique",
    "domain": "nunique"
})

print("Conflicting sentiment:", (conflicts["sentiment"] > 1).sum())
print("Conflicting domain:", (conflicts["domain"] > 1).sum())

Conflicting sentiment: 0
Conflicting domain: 0


In [ ]:
df = df.drop_duplicates(subset=["review_text"]).reset_index(drop=True)                                     ####

print("Reviews after removing duplicates:", len(df))       ###2

Reviews after removing duplicates: 7853


In [ ]:
print(df.groupby(["domain", "sentiment"]).size())

domain                sentiment
books                 negative     963
                      positive     993
dvd                   negative     959
                      positive     995
electronics           negative     979
                      positive     987
kitchen_&_housewares  negative     981
                      positive     996
dtype: int64


In [ ]:
df.to_csv("multi_domain_sentiment_clean.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df["stratify_col"] = df["domain"] + "_" + df["sentiment"]

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["stratify_col"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["stratify_col"]
)

In [ ]:
for split_df in [train_df, val_df, test_df]:
    split_df.drop(columns=["stratify_col"], inplace=True)

In [ ]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 6282
Validation: 785
Test: 786


In [ ]:
print(train_df.columns)

Index(['review_text', 'sentiment', 'domain'], dtype='object')


In [ ]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

# BERT Mini --Preliminary Baseline

In [ ]:
!pip -q install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
                                                                                                                    ####
model_name = "google/bert_uncased_L-4_H-256_A-4"             #######3

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

In [ ]:
print(model)          #### 3

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 256, padding_idx=0)
      (position_embeddings): Embedding(512, 256)
      (token_type_embeddings): Embedding(2, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=256, out_features=256, bias=True)
              (LayerNorm): LayerNorm((256,), eps=1e-12, e

# Dataset Token Statistics

In [ ]:
all_texts = pd.concat([                                                                                            ####
    train_df["review_text"],
    val_df["review_text"],
    test_df["review_text"]
])

token_counts = all_texts.apply(
    lambda text: len(tokenizer.encode(text, add_special_tokens=True))
)

print("Total tokens:", token_counts.sum())
print("Average tokens per review:", token_counts.mean())
print("Maximum tokens in one review:", token_counts.max())

Total tokens: 1354482
Average tokens per review: 172.4795619508468
Maximum tokens in one review: 4772


In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

classifier.weight
classifier.bias


In [ ]:
label_map = {
    "negative": 0,
    "positive": 1
}

train_df["labels"] = train_df["sentiment"].map(label_map)
val_df["labels"] = val_df["sentiment"].map(label_map)
test_df["labels"] = test_df["sentiment"].map(label_map)

In [ ]:
print(train_df[["sentiment", "labels"]].head())

     sentiment  labels
7284  positive       1
2993  positive       1
1040  positive       1
5775  positive       1
3635  positive       1


In [ ]:
# Tokenise the reviews before passing them to BERT.
# Reviews longer than 256 tokens are truncated to reduce training cost.
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)

Dataset({
    features: ['review_text', 'sentiment', 'domain', 'labels', '__index_level_0__'],
    num_rows: 6282
})


In [ ]:
def tokenize_function(batch):                                                                                                     ####
    return tokenizer(
        batch["review_text"],
        truncation=True,
        max_length=256                ###3
    )

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/6282 [00:00<?, ? examples/s]

Map:   0%|          | 0/785 [00:00<?, ? examples/s]

Map:   0%|          | 0/786 [00:00<?, ? examples/s]

# Dataset Columns Check

In [ ]:
print(train_dataset.column_names)

['review_text', 'sentiment', 'domain', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


# Frozen-Encoder Baseline Training **Setup**

In [ ]:
!pip -q install evaluate

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# Calculate accuracy and F1 score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]

    return {"accuracy": acc, "f1": f1}

# **Baseline Training Configuration**

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(                       ### 6
    output_dir="./baseline_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    label_names=["labels"],
    report_to="none"
)

In [ ]:
trainer = Trainer(                                    ###7
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
                                                          #####7
total_params = sum(
    p.numel() for p in model.parameters()
)

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

Trainable parameters: 514
Total parameters: 11171074


# **Baseline Results**

In [ ]:
trainer.train()                                   ####7

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.676179,0.600000,0.672234
2,0.682109,0.671915,0.605096,0.640371


TrainOutput(global_step=786, training_loss=0.6788811331790215, metrics={'train_runtime': 22.7244, 'train_samples_per_second': 552.887, 'train_steps_per_second': 34.588, 'total_flos': 61982422721352.0, 'train_loss': 0.6788811331790215, 'epoch': 2.0})

# Baseline Test **Evaluation**

In [ ]:
baseline_test_output = trainer.predict(test_dataset)
print(baseline_test_output.metrics)

{'test_loss': 0.6750493049621582, 'test_accuracy': 0.5916030534351145, 'test_f1': 0.6372881355932203, 'test_runtime': 1.0908, 'test_samples_per_second': 720.6, 'test_steps_per_second': 45.84}


# **Full Fine-Tuning (FFT)**

In [ ]:
fft_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

In [ ]:
trainable_params = sum(
    p.numel() for p in fft_model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in fft_model.parameters()
)                                                     #####8

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

Trainable parameters: 11171074
Total parameters: 11171074


In [ ]:
fft_training_args = TrainingArguments(
    output_dir="./fft_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none"
)

In [ ]:
fft_trainer = Trainer(
    model=fft_model,
    args=fft_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
import torch                                   #### 9

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


# FFT **Results**

In [ ]:
fft_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.379191,0.843312,0.842912
2,0.467167,0.361878,0.854777,0.849604


TrainOutput(global_step=786, training_loss=0.4177214450205252, metrics={'train_runtime': 53.995, 'train_samples_per_second': 232.688, 'train_steps_per_second': 14.557, 'total_flos': 61982422721352.0, 'train_loss': 0.4177214450205252, 'epoch': 2.0})

# FFT Test **Evaluation**

In [ ]:
fft_test_results = fft_trainer.evaluate(test_dataset)
fft_test_output = fft_trainer.predict(test_dataset)

print(fft_test_results)
print(fft_test_output.metrics)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.467167,0.348981,2,0.853690,0.853129


{'eval_loss': 0.34898069500923157, 'eval_accuracy': 0.8536895674300254, 'eval_f1': 0.8531289910600255}
{'test_loss': 0.34898069500923157, 'test_accuracy': 0.8536895674300254, 'test_f1': 0.8531289910600255, 'test_runtime': 0.8488, 'test_samples_per_second': 926.026, 'test_steps_per_second': 58.908}


# **LoRA (Parameter-Efficient Fine-Tuning)**

In [ ]:
!pip -q install peft

In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_model = AutoModelForSequenceClassification.from_pretrained(                                      #####
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

In [ ]:
lora_config = LoraConfig(
    r=8,                                         ####10
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"]
)

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
lora_model = get_peft_model(lora_model, lora_config)

lora_model.print_trainable_parameters()

trainable params: 33,282 || all params: 11,204,356 || trainable%: 0.2970


# LoRA Training **Configuration**

In [ ]:
lora_train_dataset = train_dataset.select_columns(
    ["input_ids", "token_type_ids", "attention_mask", "labels"]
)

lora_val_dataset = val_dataset.select_columns(
    ["input_ids", "token_type_ids", "attention_mask", "labels"]
)

lora_test_dataset = test_dataset.select_columns(
    ["input_ids", "token_type_ids", "attention_mask", "labels"]
)

print(lora_train_dataset.column_names)

['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [ ]:
lora_training_args = TrainingArguments(
    output_dir="./lora_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    label_names=["labels"],
    remove_unused_columns=False,
    report_to="none"
)

In [ ]:
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=lora_train_dataset,
    eval_dataset=lora_val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
lora_trainer.train()                       ###10

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.677896,0.598726,0.668072
2,0.687326,0.667247,0.611465,0.634731


TrainOutput(global_step=786, training_loss=0.6811407734722885, metrics={'train_runtime': 35.0388, 'train_samples_per_second': 358.574, 'train_steps_per_second': 22.432, 'total_flos': 62621911190160.0, 'train_loss': 0.6811407734722885, 'epoch': 2.0})

# LoRA Test **Evaluation**

In [ ]:
lora_test_dataset = test_dataset.remove_columns(
    ["review_text", "sentiment", "domain", "__index_level_0__"]
)

print(lora_test_dataset.column_names)

['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
lora_test_output = lora_trainer.predict(lora_test_dataset)
print(lora_test_output.metrics)

{'test_loss': 0.664121687412262, 'test_accuracy': 0.6361323155216285, 'test_f1': 0.6643192488262911, 'test_runtime': 1.9432, 'test_samples_per_second': 404.494, 'test_steps_per_second': 25.731}


# LoRA Rank **Comparison**

In [ ]:
from transformers import set_seed
from peft import LoraConfig, get_peft_model
import pandas as pd
import gc
import torch

# I got help from ChatGPT to build this function.
# I tested the code myself in Google Colab.

def run_lora_rank(rank):

    set_seed(42)

    rank_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    rank_config = LoraConfig(
        r=rank,
        lora_alpha=16,
        target_modules=["query", "value"],
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["classifier"]
    )

    rank_model = get_peft_model(rank_model, rank_config)

    trainable_params = sum(
        p.numel() for p in rank_model.parameters() if p.requires_grad
    )

    total_params = sum(
        p.numel() for p in rank_model.parameters()
    )

    print(f"\n===== LoRA Rank {rank} =====")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

    rank_training_args = TrainingArguments(
        output_dir=f"./lora_rank_{rank}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        label_names=["labels"],
        remove_unused_columns=False,
        report_to="none",
        seed=42,
        data_seed=42
    )

    rank_trainer = Trainer(
        model=rank_model,
        args=rank_training_args,
        train_dataset=lora_train_dataset,
        eval_dataset=lora_val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    train_output = rank_trainer.train()
    test_output = rank_trainer.predict(lora_test_dataset)

    result = {
        "Rank": rank,
        "Accuracy": test_output.metrics["test_accuracy"],
        "F1": test_output.metrics["test_f1"],
        "Trainable Parameters": trainable_params,
        "Trainable %": 100 * trainable_params / total_params,
        "Training Runtime (s)": train_output.metrics["train_runtime"]
    }

    del rank_trainer
    del rank_model
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
rank_results = []                                                                                   ############

for rank in [8, 16, 32]:
    result = run_lora_rank(rank)
    rank_results.append(result)
                                                    ####10
rank_results_df = pd.DataFrame(rank_results)

rank_results_df

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos


===== LoRA Rank 8 =====
Trainable parameters: 33,282
Total parameters: 11,204,356
Trainable percentage: 0.2970%


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.669976,0.617834,0.644550
2,0.682376,0.662143,0.649682,0.624829


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos


===== LoRA Rank 32 =====
Trainable parameters: 131,586
Total parameters: 11,302,660
Trainable percentage: 1.1642%


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.669886,0.628025,0.645631
2,0.682129,0.662109,0.644586,0.616231


,Rank,Accuracy,F1,Trainable Parameters,Trainable %,Training Runtime (s)
0,8,0.643766,0.623656,33282,0.297045,56.3694
1,16,0.641221,0.621984,66050,0.587784,36.4119
2,32,0.642494,0.620783,131586,1.164204,50.4979
